### Notebook For Batch Ingestion and Incremental Batch Ingestion in Databricks using Lakeflow house Connect

In [0]:
%sql 
select current_catalog();

## Setting up a new catalog using external storage and managed location

In [0]:
%sql
create external location extdbtutorial
url 'abfss://catalog-data@dbtutorialadls.dfs.core.windows.net/'
with (storage credential dbtutoiralsc);

In [0]:
%sql
create external location ext_landing_dbtutorial
url 'abfss://landing@dbtutorialadls.dfs.core.windows.net/'
with (storage credential dbtutoiralsc);

In [0]:
%sql
select current_catalog();

In [0]:
%sql
create catalog practice_catalog
managed location 'abfss://catalog-data@dbtutorialadls.dfs.core.windows.net/';

In [0]:
%sql
create schema practice_schema
managed location 'abfss://catalog-data@dbtutorialadls.dfs.core.windows.net/practice_schema';
    
create schema practice_catalog.bronze;

### CTAS and Copy INTO for Incremental Loading

In [0]:
%sql
select current_catalog(), current_schema()

In [0]:
%sql
create external volume practice_catalog.bronze.landing
location 'abfss://landing@dbtutorialadls.dfs.core.windows.net/'

Reading the data using read_file() function in Databricks

In [0]:
%sql
select *
from read_files(
    '/Volumes/practice_catalog/bronze/landing',
    format=>'parquet'
) limit 10;

NOw going to make a bronze Delta table(By default, Every table is DLT) 

In [0]:
%sql
create table if not exists practice_catalog.bronze.table1
select * 
from read_files(
    '/Volumes/practice_catalog/bronze/landing',
    format=>'parquet'
);

-- preview
select * 
from practice_catalog.bronze.table1
limit 10;

We got to know, using CTAS we can directly ingest the data from cloud storage like that. One thing to note is added _rescued_data_ column which tell us about any schema difference.

In [0]:
%sql
Describe table extended practice_catalog.bronze.table1;

For Managed table DB manages data and meta data.
For external table, DB only manage metadata, and points to the external location 

In [0]:

#using python
df = (spark
      .read
      .format('parquet').load(
        '/Volumes/practice_catalog/bronze/landing'
      )

)
display(df)

In [0]:
(df.write.mode('overwrite').SaveAsTable('practice_catalog.bronze.landing'))

### INCREMENTAL DATA INGESTION USING COPY INTO

use mergeschema for schema mismatch, It will evolve the schema

In [0]:


%sql

create schema if not exists practice_catalog.silver



In [0]:
%sql
create table if not exists practice_catalog.silver.table1;


copy into practice_catalog.silver.table1
from '/Volumes/practice_catalog/bronze/landing'
FILEFORMAT = PARQUET
copy_options = ('mergeSchema=true')

    


Copy into Read all the files at first, then while re reading, only new files will be processed, Or Better option is to use AUTO LOADER 